In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Force a fresh read from disk every time this cell runs
df = pd.read_parquet("data/market_history.parquet")
df['run_timestamp'] = pd.to_datetime(df['run_timestamp'])
df = df.sort_values(['symbol', 'run_timestamp'])

print(f"Total Rows:  {len(df)}")
print(f"Unique Runs: {df['run_timestamp'].nunique()}")
print(f"Tickers:     {sorted(df['symbol'].unique().tolist())}")
print(f"Columns:     {df.columns.tolist()}")

# Sanity checks
dupe_cols = [c for c in df.columns if c.endswith('_x') or c.endswith('_y')]
if dupe_cols:
    print(f"WARNING: Duplicate columns detected: {dupe_cols}")

missing_volume  = df['volume'].isna().sum()
missing_pct     = df['pct_change'].isna().sum()
zero_total_ups  = (df['total_ups'] == 0).sum()
print(f"Missing volume:         {missing_volume} / {len(df)}")
print(f"Missing pct_change:     {missing_pct} / {len(df)}")
print(f"Rows with 0 total_ups:  {zero_total_ups} / {len(df)}")

Total Rows:  6
Unique Runs: 1
Tickers:     ['BMNR', 'NFLX', 'NVDA', 'SNAP', 'SOFI', 'TSLA']
Columns:     ['symbol', 'price', 'volume', 'pct_change', 'mentions', 'avg_sentiment', 'avg_upvote_ratio', 'total_comments', 'total_ups', 'total_weighted_sentiment', 'total_weight', 'run_timestamp', 'sentiment', 'sentiment_volatility', 'sentiment_momentum', 'price_momentum', 'volume_momentum', 'divergence']
Missing volume:         0 / 6
Missing pct_change:     0 / 6
Rows with 0 total_ups:  2 / 6


In [2]:
# Latest run snapshot — open this in Data Wrangler for a full column view
latest_ts = df['run_timestamp'].max()
df_latest = df[df['run_timestamp'] == latest_ts].copy()

print(f"Latest run: {latest_ts}")
df_latest[['symbol', 'price', 'pct_change', 'volume', 'mentions',
           'sentiment', 'price_momentum', 'volume_momentum', 'divergence']]

Latest run: 2026-04-18 15:38:35.336459+00:00


,symbol,price,pct_change,volume,mentions,sentiment,price_momentum,volume_momentum,divergence
4,BMNR,22.95,2.27,60216000.0,1,0.964100,NaN,NaN,0
1,NFLX,97.31,-9.72,116997000.0,3,0.579625,NaN,NaN,0
0,NVDA,201.68,1.68,138518000.0,8,0.537953,NaN,NaN,0
5,SNAP,6.03,0.17,59950000.0,4,0.184540,NaN,NaN,0
3,SOFI,19.43,2.10,67365000.0,1,0.967100,NaN,NaN,0
2,TSLA,400.62,3.02,85796000.0,2,0.984897,NaN,NaN,0


In [3]:
# Divergence signals — tickers where sentiment is rising but price is falling
df_divergence = df[df['divergence'] == 1][[
    'run_timestamp', 'symbol', 'price', 'pct_change',
    'sentiment', 'sentiment_momentum', 'price_momentum', 'volume_momentum'
]].sort_values('run_timestamp', ascending=False)

print(f"Total divergence signals: {len(df_divergence)}")
df_divergence

Total divergence signals: 0


,run_timestamp,symbol,price,pct_change,sentiment,sentiment_momentum,price_momentum,volume_momentum
